# Module 1: Data Collection & Dataset Relationship Analysis
### Personalized TED Talks Recommendation System Using Machine Learning
**Infosys Springboard Virtual Internship 7.0**

---

## 1. Introduction and Objectives
Welcome to **Module 1: Data Collection** of the *Personalized TED Talks Recommendation System*. In this initial module of the machine learning lifecycle, our primary goals are:
1. **Verify Environment and Data Availability:** Inspect the project directory structure and confirm the existence of `data/ted_main.csv` and `data/transcripts.csv`.
2. **Ingest Datasets:** Load both CSV files using Pandas and verify initial records, shapes, and schemas without modifying original files.
3. **Analyze Identifiers & Candidate Keys:** Evaluate common columns and determine whether `url`, `title`, or other features can serve as a dependable primary/foreign key.
4. **Inspect Overlap & Relationships:** Quantify how many talks in the metadata have matching transcripts, diagnose discrepancies (e.g. why 86 talks lack transcripts), and assess merge viability.
5. **Document Source, Attribution, & Licensing:** Record dataset origins, licensing terms (CC BY-NC-SA 4.0), and known real-world limitations.

---
### Technology Stack for Module 1
- **Python 3.13**
- **Pandas** for structured data manipulation
- **NumPy** for numerical verification
- **Pathlib** for cross-platform filesystem handling


In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configure pandas display options
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 1000)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Pandas version: 2.3.1
NumPy version: 2.3.1


## 2. Dataset Existence Verification
We first verify that both required raw CSV files are present in the `data/` directory. If either file is missing, the code provides clear instructions on where to locate the dataset.

In [2]:
# Define paths
data_dir = Path("../data") if Path("../data").exists() else Path("data")
main_file = data_dir / "ted_main.csv"
trans_file = data_dir / "transcripts.csv"

print(f"Checking data directory: {data_dir.resolve()}")

files_status = {
    "ted_main.csv": main_file.is_file(),
    "transcripts.csv": trans_file.is_file()
}

for filename, exists in files_status.items():
    status_str = "EXISTS" if exists else "MISSING"
    print(f"  [{status_str}] {filename}")

if not all(files_status.values()):
    missing = [f for f, exists in files_status.items() if not exists]
    raise FileNotFoundError(
        f"Missing required file(s): {missing}.\n"
        f"Please place 'ted_main.csv' and 'transcripts.csv' inside '{data_dir.resolve()}'."
    )
else:
    print("\nBoth required dataset files are present and verified!")


Checking data directory: E:\infosys\TED Talks Recommendation System\data
  [EXISTS] ted_main.csv
  [EXISTS] transcripts.csv

Both required dataset files are present and verified!


## 3. Data Ingestion & Initial Inspection
We load both datasets into Pandas DataFrames and examine their shapes, columns, and sample records.
- `df_main`: Talk metadata (titles, speakers, tags, view counts, ratings, publication dates, etc.)
- `df_trans`: Spoken transcript text for TED talks

In [3]:
# Load datasets
df_main = pd.read_csv(main_file)
df_trans = pd.read_csv(trans_file)

print(f"ted_main.csv shape: {df_main.shape[0]:,} rows, {df_main.shape[1]} columns")
print(f"transcripts.csv shape: {df_trans.shape[0]:,} rows, {df_trans.shape[1]} columns")


ted_main.csv shape: 2,550 rows, 17 columns
transcripts.csv shape: 2,467 rows, 2 columns


### 3.1 Metadata Dataset (`ted_main.csv`) First 5 Rows & Schema

In [4]:
# Display first 5 rows of ted_main
df_main.head()


,comments,description,duration,event,film_date,languages,main_speaker,name,num_speaker,published_date,ratings,related_talks,speaker_occupation,tags,title,url,views
0,4553,Sir Ken Robinson makes an entertaining and profoundly mo...,1164,TED2006,1140825600,60,Ken Robinson,Ken Robinson: Do schools kill creativity?,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 19645}, {'id': 1, '...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/images/ted/1...",Author/educator,"['children', 'creativity', 'culture', 'dance', 'educatio...",Do schools kill creativity?,https://www.ted.com/talks/ken_robinson_says_schools_kill...,47227110
1,265,"With the same humor and humanity he exuded in ""An Inconv...",977,TED2006,1140825600,43,Al Gore,Al Gore: Averting the climate crisis,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 544}, {'id': 3, 'na...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/images/ted/5...",Climate advocate,"['alternative energy', 'cars', 'climate change', 'cultur...",Averting the climate crisis,https://www.ted.com/talks/al_gore_on_averting_climate_cr...,3200520
2,124,New York Times columnist David Pogue takes aim at techno...,1286,TED2006,1140739200,26,David Pogue,David Pogue: Simplicity sells,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 964}, {'id': 3, 'na...","[{'id': 1725, 'hero': 'https://pe.tedcdn.com/images/ted/...",Technology columnist,"['computers', 'entertainment', 'interface design', 'medi...",Simplicity sells,https://www.ted.com/talks/david_pogue_says_simplicity_se...,1636292
3,200,"In an emotionally charged talk, MacArthur-winning activi...",1116,TED2006,1140912000,35,Majora Carter,Majora Carter: Greening the ghetto,1,1151367060,"[{'id': 3, 'name': 'Courageous', 'count': 760}, {'id': 1...","[{'id': 1041, 'hero': 'https://pe.tedcdn.com/images/ted/...",Activist for environmental justice,"['MacArthur grant', 'activism', 'business', 'cities', 'e...",Greening the ghetto,https://www.ted.com/talks/majora_carter_s_tale_of_urban_...,1697550
4,593,You've never seen data presented like this. With the dra...,1190,TED2006,1140566400,48,Hans Rosling,Hans Rosling: The best stats you've ever seen,1,1151440680,"[{'id': 9, 'name': 'Ingenious', 'count': 3202}, {'id': 7...","[{'id': 2056, 'hero': 'https://pe.tedcdn.com/images/ted/...",Global health expert; data visionary,"['Africa', 'Asia', 'Google', 'demo', 'economics', 'globa...",The best stats you've ever seen,https://www.ted.com/talks/hans_rosling_shows_the_best_st...,12005869


In [5]:
# Display column data types and non-null counts for ted_main
df_main.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2550 entries, 0 to 2549
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   comments            2550 non-null   int64 
 1   description         2550 non-null   object
 2   duration            2550 non-null   int64 
 3   event               2550 non-null   object
 4   film_date           2550 non-null   int64 
 5   languages           2550 non-null   int64 
 6   main_speaker        2550 non-null   object
 7   name                2550 non-null   object
 8   num_speaker         2550 non-null   int64 
 9   published_date      2550 non-null   int64 
 10  ratings             2550 non-null   object
 11  related_talks       2550 non-null   object
 12  speaker_occupation  2544 non-null   object
 13  tags                2550 non-null   object
 14  title               2550 non-null   object
 15  url                 2550 non-null   object
 16  views               2550

### 3.2 Transcripts Dataset (`transcripts.csv`) First 5 Rows & Schema

In [6]:
# Display first 5 rows of transcripts
df_trans.head()


,transcript,url
0,"Good morning. How are you?(Laughter)It's been great, has...",https://www.ted.com/talks/ken_robinson_says_schools_kill...
1,"Thank you so much, Chris. And it's truly a great honor t...",https://www.ted.com/talks/al_gore_on_averting_climate_cr...
2,"(Music: ""The Sound of Silence,"" Simon & Garfunkel)Hello ...",https://www.ted.com/talks/david_pogue_says_simplicity_se...
3,If you're here today — and I'm very happy that you are —...,https://www.ted.com/talks/majora_carter_s_tale_of_urban_...
4,"About 10 years ago, I took on the task to teach global d...",https://www.ted.com/talks/hans_rosling_shows_the_best_st...


In [7]:
# Display column data types and non-null counts for transcripts
df_trans.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   transcript  2467 non-null   object
 1   url         2467 non-null   object
dtypes: object(2)
memory usage: 38.7+ KB


## 4. Identifier Analysis & Candidate Key Evaluation
To join talk metadata with transcript text, we need a reliable common identifier.
Let us examine:
1. Common column names across both datasets.
2. Uniqueness of candidate keys (`url`, `name`, `title`).
3. Whitespace / newline formatting quirks.

In [8]:
# Identify intersecting column names
common_columns = list(set(df_main.columns).intersection(set(df_trans.columns)))
print(f"Common columns between datasets: {common_columns}")

# Inspect uniqueness in metadata
print(f"Total records in ted_main: {len(df_main):,}")
print(f"Unique 'url' values in ted_main: {df_main['url'].nunique():,}")
print(f"Unique 'title' values in ted_main: {df_main['title'].nunique():,}")
print(f"Unique 'name' values in ted_main: {df_main['name'].nunique():,}")

# Inspect uniqueness in transcripts
print(f"\nTotal records in transcripts: {len(df_trans):,}")
print(f"Unique 'url' values in transcripts: {df_trans['url'].nunique():,}")


Common columns between datasets: ['url']
Total records in ted_main: 2,550
Unique 'url' values in ted_main: 2,550
Unique 'title' values in ted_main: 2,550
Unique 'name' values in ted_main: 2,550

Total records in transcripts: 2,467
Unique 'url' values in transcripts: 2,464


### 4.1 Diagnosing URL Whitespace and Duplicate Transcripts
Notice that `transcripts.csv` has 2,467 rows but only 2,464 unique URLs (3 duplicate URLs).
Furthermore, let us check whether the URLs contain whitespace or newline (`\n`) characters.

In [9]:
# Check sample URL character representation
sample_main_url = df_main['url'].iloc[0]
sample_trans_url = df_trans['url'].iloc[0]
print("Sample ted_main URL representation:", repr(sample_main_url))
print("Sample transcripts URL representation:", repr(sample_trans_url))

# Count trailing newlines
main_has_newline = df_main['url'].str.endswith('\n').sum()
trans_has_newline = df_trans['url'].str.endswith('\n').sum()
print(f"ted_main URLs ending with '\\n': {main_has_newline} of {len(df_main)}")
print(f"transcripts URLs ending with '\\n': {trans_has_newline} of {len(df_trans)}")

# Inspect the duplicate URL records in transcripts.csv
clean_trans_urls = df_trans['url'].str.strip()
dup_mask = clean_trans_urls.duplicated(keep=False)
dup_records = df_trans[dup_mask].copy()
dup_records['clean_url'] = clean_trans_urls[dup_mask]
print(f"\nDuplicate transcript rows found: {len(dup_records)}")
print(dup_records[['clean_url']].drop_duplicates())


Sample ted_main URL representation: 'https://www.ted.com/talks/ken_robinson_says_schools_kill_creativity\n'
Sample transcripts URL representation: 'https://www.ted.com/talks/ken_robinson_says_schools_kill_creativity\n'
ted_main URLs ending with '\n': 2550 of 2550
transcripts URLs ending with '\n': 2467 of 2467

Duplicate transcript rows found: 6
                                                        clean_url
1107  https://www.ted.com/talks/jonathan_haidt_humanity_s_stai...
1108        https://www.ted.com/talks/rob_reid_the_8_billion_ipod
1109     https://www.ted.com/talks/brene_brown_listening_to_shame


In [10]:
# Verify whether duplicate transcript rows are 100% identical
for url, group in dup_records.groupby('clean_url'):
    t0 = group['transcript'].iloc[0]
    t1 = group['transcript'].iloc[1]
    is_identical = (t0 == t1)
    print(f"URL: {url.split('/')[-1]}")
    print(f"  Length: {len(t0):,} characters | Identical text: {is_identical}")


URL: brene_brown_listening_to_shame
  Length: 15,369 characters | Identical text: True
URL: jonathan_haidt_humanity_s_stairway_to_self_transcendence
  Length: 16,107 characters | Identical text: True
URL: rob_reid_the_8_billion_ipod
  Length: 4,147 characters | Identical text: True


## 5. Dataset Overlap & Relationship Inspection
With normalized URLs (stripped of trailing `\n`), let us analyze the exact overlap between `ted_main.csv` and `transcripts.csv`.

In [11]:
main_urls = set(df_main['url'].str.strip())
trans_urls = set(df_trans['url'].str.strip())

common_urls = main_urls.intersection(trans_urls)
main_only_urls = main_urls - trans_urls
trans_only_urls = trans_urls - main_urls

print(f"Total unique talks in ted_main: {len(main_urls):,}")
print(f"Total unique talks in transcripts: {len(trans_urls):,}")
print(f"Common talks present in both datasets: {len(common_urls):,} ({len(common_urls)/len(main_urls):.2%})")
print(f"Talks in ted_main without transcripts: {len(main_only_urls):,} ({len(main_only_urls)/len(main_urls):.2%})")
print(f"Transcripts without metadata: {len(trans_only_urls):,} ({len(trans_only_urls)/len(trans_urls):.2%})")


Total unique talks in ted_main: 2,550
Total unique talks in transcripts: 2,464
Common talks present in both datasets: 2,464 (96.63%)
Talks in ted_main without transcripts: 86 (3.37%)
Transcripts without metadata: 0 (0.00%)


### 5.1 Deep-Dive: Why do 86 talks lack transcripts?
Let us inspect sample talks from `ted_main.csv` that do not have matching transcripts in `transcripts.csv`.

In [12]:
missing_transcripts_df = df_main[df_main['url'].str.strip().isin(main_only_urls)]
print("Sample talks without transcripts:")
missing_transcripts_df[['title', 'main_speaker', 'speaker_occupation', 'tags']].head(10)


Sample talks without transcripts:


,title,main_speaker,speaker_occupation,tags
58,"A dance of ""Symbiosis""",Pilobolus,Dance company,"['dance', 'entertainment', 'nature', 'performance', 'sci..."
115,"A string quartet plays ""Blue Room""",Ethel,String quartet,"['cello', 'collaboration', 'culture', 'entertainment', '..."
135,"""Woza""",Vusi Mahlasela,"Musician, activist","['Africa', 'entertainment', 'guitar', 'live music', 'mus..."
209,"""M'Bifo""",Rokia Traore,Singer-songwriter,"['Africa', 'entertainment', 'guitar', 'live music', 'mus..."
237,"""Kounandi""",Rokia Traore,Singer-songwriter,"['Africa', 'guitar', 'live music', 'music', 'singer']"
249,"A performance with breath, music, passion",Sxip Shirey + Rachelle Garniez,Musician,"['entertainment', 'live music', 'music']"
399,Piano jazz that rocks,Eric Lewis,Pianist,"['entertainment', 'innovation', 'invention', 'live music..."
446,Chaos and harmony on piano,Eric Lewis,Pianist,"['art', 'entertainment', 'live music', 'music', 'piano']"
474,An electrifying organ performance,Qi Zhang,Organist,"['TEDx', 'china', 'music', 'performance', 'piano', 'tech..."
512,Hypnotic South Asian improv music,Vishal Vaid,Musician,"['Asia', 'beauty', 'culture', 'history', 'music', 'perfo..."


**Domain Finding:**
Notice the titles and tags of talks without transcripts:
- *"A dance of 'Symbiosis'"* (dance, performance)
- *"A string quartet plays 'Blue Room'"* (live music, cello)
- *"'Woza'"* (African guitar music)
- *"Piano jazz that rocks"* (instrumental piano)
- *"An electrifying organ performance"* (organ performance)

These 86 talks are **instrumental musical performances, choreographies, and artistic demonstrations**. Because there was no spoken speech, official TED transcripts were naturally not created. This is an authentic domain attribute rather than corrupt or missing data.

## 6. Dataset Documentation, Source, Licensing, & Limitations

| Attribute | Details |
| :--- | :--- |
| **Dataset Source** | Kaggle TED Talks Dataset (`rounakbanik/ted-talks`), scraped from official TED website ([ted.com](https://www.ted.com)). |
| **Curator / Attribution** | Rounak Banik; TED Conferences LLC. |
| **Licensing** | Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0) & TED Terms of Use for non-commercial educational use. |
| **Temporal Coverage** | TED Talks filmed from May 1972 through August 2017, and published on ted.com through September 21, 2017. |
| **Metadata Features (`ted_main.csv`)** | 2,550 talks, 17 columns: `comments`, `description`, `duration`, `event`, `film_date`, `languages`, `main_speaker`, `name`, `num_speaker`, `published_date`, `ratings`, `related_talks`, `speaker_occupation`, `tags`, `title`, `url`, `views`. |
| **Transcript Features (`transcripts.csv`)** | 2,467 rows (2,464 unique talks after deduplication), 2 columns: `transcript`, `url`. |
| **Reliable Linking Key** | `url` (after stripping trailing `\n` newline characters). |
| **Known Limitations** | 1. **Static Snapshot:** Data is capped up to September 2017.<br>2. **No Per-User Interactions:** Lacks individual user watch histories or per-user rating profiles (ratings are aggregate community votes), requiring collaborative filtering to use alternative or simulated interaction matrices.<br>3. **86 Non-Spoken Talks:** Instrumental music, dance, and audio-visual art talks lack transcript text, so content recommendations for these items must rely on `title`, `description`, and `tags`. |


## 7. Summary & Next Steps
- **Data Collection (Module 1) is complete.**
- Both datasets have been safely loaded, audited, and mapped via `url`.
- Trailing `\n` formatting and 3 duplicate transcript rows were diagnosed.
- Proceed to **Module 2 (`02_eda_preprocessing.ipynb`)** for exploratory data analysis, data quality reporting, and end-to-end cleaning.